# Clustering Analysis of Colorado Bird Sightings

This notebook uses clustering analysis on `birdsong_df` to answer three questions:

1. What are the bird hotspots in Colorado during each of the four seasons?
2. How has the prevalence of different species of birds changed over time?
3. What are the typical flight patterns of different bird species?

The notebook also captures dataset snapshots before and after each major transformation, documents model assumptions and tuning choices, and evaluates clustering quality with Silhouette Score and Davies-Bouldin Index.

## Why K-Means Clustering Was Chosen

`birdsong_df` is largely unlabeled for the questions we want to answer. We do not have a ground-truth label that says whether a county is a hotspot, which species belong to the same temporal prevalence profile, or which species share similar flight behavior. That makes unsupervised learning the right modeling family.

K-Means is used here because:

- The transformed feature sets are numeric and can be standardized.
- We want interpretable centroids that summarize each cluster.
- The dataset is large enough that a fast centroid-based method is practical.
- We can tune the number of clusters using internal validation metrics.

## Model Assumptions

K-Means assumes:

- Clusters are reasonably compact and separable in feature space.
- Euclidean distance is meaningful after feature scaling.
- Features with larger raw units should not dominate, so scaling is required.
- The chosen number of clusters `k` is not known in advance and must be tuned.

For the flight-pattern analysis, an additional practical assumption is made: because the dataset contains observations rather than true tracked trajectories, flight patterns are approximated using seasonal and monthly geographic movement signatures rather than literal path reconstruction.

In [161]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import davies_bouldin_score, silhouette_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
sns.set_theme(style='whitegrid', context='talk')

In [162]:
# Use the in-memory DataFrame when it already exists; otherwise fall back to the saved CSV.
if 'birdsong_df' not in globals():
    birdsong_df = pd.read_csv('..\\data\\birdsong.csv')

birdsong_df = birdsong_df.copy()
birdsong_df.head()

,common_name,date,bird_count,county,family,total_monthly_precipitation,monthly_max_temp,monthly_min_temp,urban_population_level,county_total_population,is_fire,is_flood,bird_count_adjusted_for_population,season,year,month,log_bird_count
0,Band-tailed Pigeon,2021-01-01,1.0000,Garfield,Pigeons and Doves,23.0158,1.4122,-10.4589,4,"62,178.4800",0,0,0.0000,Winter,2021,1,0.6931
1,Pine Warbler,2021-01-01,1.0000,Boulder,New World Warblers,9.8737,3.8300,-8.2778,5,"333,404.0640",0,0,0.0000,Winter,2021,1,0.6931
2,White-winged Scoter,2021-01-01,1.0000,Larimer,"Ducks, Geese, and Waterfowl",10.8259,1.8541,-10.4572,5,"361,938.5280",0,0,0.0000,Winter,2021,1,0.6931
3,Brown Thrasher,2021-01-01,1.0000,Boulder,Mockingbirds and Thrashers,9.8737,3.8300,-8.2778,5,"333,404.0640",0,0,0.0000,Winter,2021,1,0.6931
4,Bonaparte's Gull,2021-01-01,2.0000,Pueblo,"Gulls, Terns, and Skimmers",14.1278,8.5667,-8.4000,5,"169,507.2960",0,0,0.0000,Winter,2021,1,1.0986


## Snapshot Before Transformations

In [163]:
print(birdsong_df.shape)
birdsong_df.info()
birdsong_df.head(10)

(349430, 17)
<class 'pandas.DataFrame'>
RangeIndex: 349430 entries, 0 to 349429
Data columns (total 17 columns):
 #   Column                              Non-Null Count   Dtype         
---  ------                              --------------   -----         
 0   common_name                         349430 non-null  str           
 1   date                                349430 non-null  datetime64[us]
 2   bird_count                          349430 non-null  float64       
 3   county                              349430 non-null  str           
 4   family                              349430 non-null  str           
 5   total_monthly_precipitation         349430 non-null  float64       
 6   monthly_max_temp                    349430 non-null  float64       
 7   monthly_min_temp                    349430 non-null  float64       
 8   urban_population_level              349430 non-null  int64         
 9   county_total_population             349430 non-null  float64       
 10  is_fir

,common_name,date,bird_count,county,family,total_monthly_precipitation,monthly_max_temp,monthly_min_temp,urban_population_level,county_total_population,is_fire,is_flood,bird_count_adjusted_for_population,season,year,month,log_bird_count
0,Band-tailed Pigeon,2021-01-01,1.0000,Garfield,Pigeons and Doves,23.0158,1.4122,-10.4589,4,"62,178.4800",0,0,0.0000,Winter,2021,1,0.6931
1,Pine Warbler,2021-01-01,1.0000,Boulder,New World Warblers,9.8737,3.8300,-8.2778,5,"333,404.0640",0,0,0.0000,Winter,2021,1,0.6931
2,White-winged Scoter,2021-01-01,1.0000,Larimer,"Ducks, Geese, and Waterfowl",10.8259,1.8541,-10.4572,5,"361,938.5280",0,0,0.0000,Winter,2021,1,0.6931
3,Brown Thrasher,2021-01-01,1.0000,Boulder,Mockingbirds and Thrashers,9.8737,3.8300,-8.2778,5,"333,404.0640",0,0,0.0000,Winter,2021,1,0.6931
4,Bonaparte's Gull,2021-01-01,2.0000,Pueblo,"Gulls, Terns, and Skimmers",14.1278,8.5667,-8.4000,5,"169,507.2960",0,0,0.0000,Winter,2021,1,1.0986
5,American Three-toed Woodpecker,2021-01-01,1.0000,Larimer,Woodpeckers,10.8259,1.8541,-10.4572,5,"361,938.5280",0,0,0.0000,Winter,2021,1,0.6931
6,Greater Roadrunner,2021-01-01,1.0000,Pueblo,Cuckoos,14.1278,8.5667,-8.4000,5,"169,507.2960",0,0,0.0000,Winter,2021,1,0.6931
7,Graylag x Swan Goose (hybrid),2021-01-01,3.0000,Denver,"Ducks, Geese, and Waterfowl",6.0800,9.3900,-5.9000,5,"721,246.1760",0,0,0.0000,Winter,2021,1,1.3863
8,Black Phoebe,2021-01-01,1.0000,Pueblo,Tyrant Flycatchers,14.1278,8.5667,-8.4000,5,"169,507.2960",0,0,0.0000,Winter,2021,1,0.6931
9,Northern Mockingbird,2021-01-01,1.0000,Otero,Mockingbirds and Thrashers,14.8273,7.9229,-7.6471,3,"18,839.5200",0,0,0.0001,Winter,2021,1,0.6931


## Core Transformations

The clustering tasks need a consistent temporal format, numeric features, and stable prevalence measures. The major steps below standardize dates, create time-derived features, and build analysis-specific feature tables.

In [164]:
season_map = {
    12: 'Winter', 1: 'Winter', 2: 'Winter',
    3: 'Spring', 4: 'Spring', 5: 'Spring',
    6: 'Summer', 7: 'Summer', 8: 'Summer',
    9: 'Autumn', 10: 'Autumn', 11: 'Autumn'
}

birdsong_df['date'] = pd.to_datetime(birdsong_df['date'].astype(str))
birdsong_df['year'] = birdsong_df['date'].dt.year
birdsong_df['month'] = birdsong_df['date'].dt.month
birdsong_df['season'] = birdsong_df['month'].map(season_map)
birdsong_df['bird_count'] = pd.to_numeric(birdsong_df['bird_count'], errors='coerce').fillna(1)

if 'bird_count_adjusted_for_population' not in birdsong_df.columns:
    birdsong_df['county_total_population'] = pd.to_numeric(
        birdsong_df.get('county_total_population', np.nan), errors='coerce'
    )
    birdsong_df['bird_count_adjusted_for_population'] = (
        birdsong_df['bird_count'] / birdsong_df['county_total_population']
    )

birdsong_df['bird_count_adjusted_for_population'] = birdsong_df['bird_count_adjusted_for_population'].replace([np.inf, -np.inf], np.nan)
birdsong_df['log_bird_count'] = np.log1p(birdsong_df['bird_count'])

birdsong_df[['common_name', 'date', 'year', 'month', 'season', 'bird_count', 'bird_count_adjusted_for_population', 'log_bird_count']].head(10)

,common_name,date,year,month,season,bird_count,bird_count_adjusted_for_population,log_bird_count
0,Band-tailed Pigeon,2021-01-01,2021,1,Winter,1.0000,0.0000,0.6931
1,Pine Warbler,2021-01-01,2021,1,Winter,1.0000,0.0000,0.6931
2,White-winged Scoter,2021-01-01,2021,1,Winter,1.0000,0.0000,0.6931
3,Brown Thrasher,2021-01-01,2021,1,Winter,1.0000,0.0000,0.6931
4,Bonaparte's Gull,2021-01-01,2021,1,Winter,2.0000,0.0000,1.0986
5,American Three-toed Woodpecker,2021-01-01,2021,1,Winter,1.0000,0.0000,0.6931
6,Greater Roadrunner,2021-01-01,2021,1,Winter,1.0000,0.0000,0.6931
7,Graylag x Swan Goose (hybrid),2021-01-01,2021,1,Winter,3.0000,0.0000,1.3863
8,Black Phoebe,2021-01-01,2021,1,Winter,1.0000,0.0000,0.6931
9,Northern Mockingbird,2021-01-01,2021,1,Winter,1.0000,0.0001,0.6931


### Snapshot After Base Feature Engineering

In [165]:
birdsong_df[['common_name', 'county', 'date', 'season', 'bird_count', 'bird_count_adjusted_for_population', 'monthly_max_temp', 'monthly_min_temp', 'total_monthly_precipitation']].head(10)

,common_name,county,date,season,bird_count,bird_count_adjusted_for_population,monthly_max_temp,monthly_min_temp,total_monthly_precipitation
0,Band-tailed Pigeon,Garfield,2021-01-01,Winter,1.0000,0.0000,1.4122,-10.4589,23.0158
1,Pine Warbler,Boulder,2021-01-01,Winter,1.0000,0.0000,3.8300,-8.2778,9.8737
2,White-winged Scoter,Larimer,2021-01-01,Winter,1.0000,0.0000,1.8541,-10.4572,10.8259
3,Brown Thrasher,Boulder,2021-01-01,Winter,1.0000,0.0000,3.8300,-8.2778,9.8737
4,Bonaparte's Gull,Pueblo,2021-01-01,Winter,2.0000,0.0000,8.5667,-8.4000,14.1278
5,American Three-toed Woodpecker,Larimer,2021-01-01,Winter,1.0000,0.0000,1.8541,-10.4572,10.8259
6,Greater Roadrunner,Pueblo,2021-01-01,Winter,1.0000,0.0000,8.5667,-8.4000,14.1278
7,Graylag x Swan Goose (hybrid),Denver,2021-01-01,Winter,3.0000,0.0000,9.3900,-5.9000,6.0800
8,Black Phoebe,Pueblo,2021-01-01,Winter,1.0000,0.0000,8.5667,-8.4000,14.1278
9,Northern Mockingbird,Otero,2021-01-01,Winter,1.0000,0.0001,7.9229,-7.6471,14.8273


## Shared Utilities

Hyperparameter tuning is handled by testing multiple values of `k` and selecting the value that maximizes Silhouette Score, with Davies-Bouldin Index used as a secondary quality check. This is appropriate because the tasks are unsupervised and do not have true labels.

In [166]:
# Test over multiple values of k to find the one that gives the best evaluation score
def evaluate_kmeans_grid(X, k_values=range(2, 9), random_state=42):
    rows = []
    for k in k_values:
        model = KMeans(n_clusters=k, random_state=random_state, n_init=20)
        labels = model.fit_predict(X)
        rows.append({
            'k': k,
            'silhouette_score': silhouette_score(X, labels),
            'davies_bouldin_index': davies_bouldin_score(X, labels)
        })
    scores = pd.DataFrame(rows)
    best_k = scores.sort_values(['silhouette_score', 'davies_bouldin_index'], ascending=[False, True]).iloc[0]['k']
    return scores, int(best_k)

# Create a model with the best k value and fit the data to it
def fit_final_kmeans(X, best_k, random_state=42):
    model = KMeans(n_clusters=best_k, random_state=random_state, n_init=20)
    labels = model.fit_predict(X)  # cluster labels are numbers 1, 2, ..., k
    metrics = {
        'silhouette_score': silhouette_score(X, labels),
        'davies_bouldin_index': davies_bouldin_score(X, labels)
    }
    return model, labels, metrics

## 1. Bird Hotspots by Season

A hotspot is modeled here as a county-season combination with similar abundance, diversity, and environmental conditions. County-level aggregation is appropriate because the question asks for regional hotspots rather than individual sightings.

In [167]:
hotspot_features = (
    birdsong_df
    .groupby(['season', 'county'], as_index=False)
    .agg(
        total_bird_count=('bird_count', 'sum'),
        avg_bird_count=('bird_count', 'mean'),
        adjusted_bird_count=('bird_count_adjusted_for_population', 'mean'),
        species_richness=('common_name', 'nunique'),
        avg_precipitation=('total_monthly_precipitation', 'mean'),
        avg_max_temp=('monthly_max_temp', 'mean'),
        avg_min_temp=('monthly_min_temp', 'mean'),
        avg_population=('county_total_population', 'mean'),
        fire_rate=('is_fire', 'mean'),
        flood_rate=('is_flood', 'mean')
    )
)

hotspot_features.head(10)

,season,county,total_bird_count,avg_bird_count,adjusted_bird_count,species_richness,avg_precipitation,avg_max_temp,avg_min_temp,avg_population,fire_rate,flood_rate
0,Autumn,Adams,"10,708.0000",9.3848,0.0000,227,15.7197,22.9850,5.0132,"532,094.3122",0.0000,0.0000
1,Autumn,Alamosa,"2,087.0000",6.4613,0.0004,127,20.5382,20.7467,1.0048,"16,747.5545",0.0000,0.0000
2,Autumn,Arapahoe,"58,368.0000",7.2914,0.0000,292,22.4786,20.7356,4.7098,"673,231.8215",0.0000,0.0000
3,Autumn,Archuleta,"1,342.0000",3.2260,0.0002,129,44.7430,20.2064,4.8084,"13,665.4999",0.0000,0.0000
4,Autumn,Baca,"1,542.0000",2.7340,0.0008,176,21.5782,23.7662,6.0375,"3,583.3067",0.0000,0.0000
5,Autumn,Bent,"8,716.0000",10.8273,0.0019,208,16.4777,25.1373,5.1485,"5,766.4270",0.0000,0.0000
6,Autumn,Boulder,"106,534.0000",10.9558,0.0000,336,22.0492,16.4570,1.7042,"338,342.6022",0.0000,0.0000
7,Autumn,Broomfield,"10,466.0000",4.2736,0.0001,183,20.9944,20.0744,3.3209,"76,304.2841",0.0000,0.0000
8,Autumn,Chaffee,"4,690.0000",5.3054,0.0003,189,25.1782,15.1175,-0.8366,"19,933.0265",0.0000,0.0000
9,Autumn,Cheyenne,"4,750.0000",12.4346,0.0069,112,17.0789,25.7296,7.1963,"1,805.0577",0.0000,0.0000


### Snapshot Before Hotspot Scaling

In [168]:
hotspot_features.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
season,256,4,Autumn,64,NaN,NaN,NaN,NaN,NaN,NaN,NaN
county,256,64,Adams,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
total_bird_count,256.0000,NaN,NaN,NaN,"9,640.3828","17,324.5717",64.0000,"1,045.5000","2,793.0000","8,367.5000","106,534.0000"
avg_bird_count,256.0000,NaN,NaN,NaN,11.4188,28.2083,1.5864,3.1456,4.6454,7.8851,316.7348
adjusted_bird_count,256.0000,NaN,NaN,NaN,0.0018,0.0054,0.0000,0.0001,0.0003,0.0013,0.0607
species_richness,256.0000,NaN,NaN,NaN,136.3008,78.7060,9.0000,79.7500,128.0000,181.2500,363.0000
avg_precipitation,256.0000,NaN,NaN,NaN,37.9501,18.4466,5.3338,22.0000,37.4326,51.6490,91.3747
avg_max_temp,256.0000,NaN,NaN,NaN,16.2628,9.1412,-2.4093,9.2158,16.5480,22.9825,33.4594
avg_min_temp,256.0000,NaN,NaN,NaN,0.5752,7.9933,-16.4944,-6.3259,1.0491,5.6284,16.1676
avg_population,256.0000,NaN,NaN,NaN,"92,411.4907","186,857.0988",717.3612,"5,907.2869","15,503.4420","45,341.5386","749,437.1783"


In [169]:
hotspot_numeric_cols = [
    'total_bird_count', 'avg_bird_count', 'adjusted_bird_count', 'species_richness',
    'avg_precipitation', 'avg_max_temp', 'avg_min_temp', 'avg_population',
    'fire_rate', 'flood_rate'
]

hotspot_scaled = hotspot_features.copy()
hotspot_scaled[hotspot_numeric_cols] = StandardScaler().fit_transform(hotspot_scaled[hotspot_numeric_cols])
hotspot_scaled.head(10)

,season,county,total_bird_count,avg_bird_count,adjusted_bird_count,species_richness,avg_precipitation,avg_max_temp,avg_min_temp,avg_population,fire_rate,flood_rate
0,Autumn,Adams,0.0617,-0.0722,-0.3236,1.1546,-1.2075,0.7368,0.5563,2.3577,-0.1357,-0.1896
1,Autumn,Alamosa,-0.4368,-0.1761,-0.2553,-0.1184,-0.9458,0.4915,0.0538,-0.4057,-0.1357,-0.1896
2,Autumn,Arapahoe,2.8181,-0.1466,-0.3249,1.9821,-0.8404,0.4903,0.5183,3.1145,-0.1357,-0.1896
3,Autumn,Archuleta,-0.4799,-0.2910,-0.2831,-0.0929,0.3690,0.4323,0.5306,-0.4222,-0.1357,-0.1896
4,Autumn,Baca,-0.4684,-0.3085,-0.1849,0.5054,-0.8893,0.8225,0.6847,-0.4763,-0.1357,-0.1896
5,Autumn,Bent,-0.0535,-0.0210,0.0231,0.9128,-1.1663,0.9727,0.5733,-0.4646,-0.1357,-0.1896
6,Autumn,Boulder,5.6038,-0.0164,-0.3209,2.5423,-0.8637,0.0213,0.1415,1.3187,-0.1357,-0.1896
7,Autumn,Broomfield,0.0477,-0.2538,-0.3165,0.5945,-0.9210,0.4178,0.3442,-0.0864,-0.1357,-0.1896
8,Autumn,Chaffee,-0.2863,-0.2171,-0.2774,0.6709,-0.6937,-0.1255,-0.1770,-0.3886,-0.1357,-0.1896
9,Autumn,Cheyenne,-0.2828,0.0361,0.9485,-0.3094,-1.1337,1.0377,0.8300,-0.4858,-0.1357,-0.1896


### Snapshot After Hotspot Scaling

In [170]:
# Note: Scaling creates negative values from originally positive values
hotspot_scaled.describe().T

,count,mean,std,min,25%,50%,75%,max
total_bird_count,256.0000,-0.0000,1.0020,-0.5538,-0.4971,-0.3960,-0.0736,5.6038
avg_bird_count,256.0000,0.0000,1.0020,-0.3492,-0.2939,-0.2406,-0.1255,10.8448
adjusted_bird_count,256.0000,-0.0000,1.0020,-0.3261,-0.3102,-0.2642,-0.0855,10.9717
species_richness,256.0000,0.0000,1.0020,-1.6206,-0.7199,-0.1057,0.5722,2.8860
avg_precipitation,256.0000,-0.0000,1.0020,-1.7716,-0.8664,-0.0281,0.7441,2.9018
avg_max_temp,256.0000,0.0000,1.0020,-2.0466,-0.7724,0.0313,0.7365,1.8849
avg_min_temp,256.0000,-0.0000,1.0020,-2.1397,-0.8651,0.0594,0.6334,1.9545
avg_population,256.0000,-0.0000,1.0020,-0.4917,-0.4638,-0.4124,-0.2524,3.5231
fire_rate,256.0000,-0.0000,1.0020,-0.1357,-0.1357,-0.1357,-0.1357,9.5926
flood_rate,256.0000,0.0000,1.0020,-0.1896,-0.1896,-0.1896,-0.1896,11.6543


In [171]:
# Store counties identified as hotspots
season_hotspot_results = []
# Store evaluation results for each season
season_hotspot_metrics = []

for season_name, season_df in hotspot_scaled.groupby('season'):
    X = season_df[hotspot_numeric_cols]
    scores, best_k = evaluate_kmeans_grid(X, k_values=range(2, min(8, len(season_df) - 1) + 1))
    model, labels, metrics = fit_final_kmeans(X, best_k)

    clustered = season_df.copy()
    clustered['cluster'] = labels
    # Each cluster is defined by its centroid, which is computed by the average features values for each cluster
    centroids = clustered.groupby('cluster')[hotspot_numeric_cols].mean()
    # Picks the cluster that looks most like a hotspot by finding the cluster with the highest average values across total_bird_count, species_richness, and adjusted_bird_count
    hotspot_cluster = centroids[['total_bird_count', 'species_richness', 'adjusted_bird_count']].mean(axis=1).idxmax()
    hotspots = clustered[clustered['cluster'] == hotspot_cluster].copy()
    hotspots = hotspots.sort_values(['total_bird_count', 'species_richness'], ascending=False)

    metrics_row = {
        'season': season_name,
        'best_k': best_k,
        'silhouette_score': metrics['silhouette_score'],
        'davies_bouldin_index': metrics['davies_bouldin_index']
    }

    season_hotspot_results.append(hotspots)
    season_hotspot_metrics.append(metrics_row)

season_hotspot_metrics_df = pd.DataFrame(season_hotspot_metrics).sort_values('season')
season_hotspot_metrics_df

,season,best_k,silhouette_score,davies_bouldin_index
0,Autumn,2,0.6125,0.7601
1,Spring,3,0.4686,0.6376
2,Summer,3,0.5748,0.7549
3,Winter,2,0.7216,0.5922


In [172]:
seasonal_hotspot_summary = (
    pd.concat(season_hotspot_results, ignore_index=True)
    [['season', 'county', 'cluster', 'total_bird_count', 'species_richness', 'adjusted_bird_count']]
    .sort_values(['season', 'total_bird_count', 'species_richness'], ascending=[True, False, False])
)

seasonal_hotspot_summary.groupby('season').head(5)

,season,county,cluster,total_bird_count,species_richness,adjusted_bird_count
0,Autumn,Boulder,1,5.6038,2.5423,-0.3209
1,Autumn,Logan,1,5.3232,1.0146,0.2487
2,Autumn,Denver,1,4.1438,1.0401,-0.3204
3,Autumn,Larimer,1,2.8401,2.6696,-0.3236
4,Autumn,Arapahoe,1,2.8181,1.9821,-0.3249
9,Spring,Conejos,2,0.9997,-0.8950,4.5982
10,Summer,Boulder,2,1.5977,1.9439,-0.3252
11,Summer,Larimer,2,1.2621,2.0203,-0.3253
12,Summer,Jefferson,2,0.7381,1.6129,-0.3261
13,Winter,Kit Carson,1,2.7580,-0.9586,7.6819


The table above answers the seasonal hotspot question by identifying the counties that fall into the highest-abundance and highest-diversity cluster for each season.

## 2. Species Prevalence Changes Over Time

To study how species prevalence changes over time, each species is represented by a monthly prevalence profile. Prevalence is defined as total observed count per month, and a small `log1p` transform is applied to reduce the influence of unusually large counts.

In [173]:
species_monthly = (
    birdsong_df
    .groupby(['common_name', 'date'], as_index=False)
    .agg(monthly_bird_count=('bird_count', 'sum'))
)
species_monthly['log_monthly_bird_count'] = np.log1p(species_monthly['monthly_bird_count'])
species_monthly.head(10)

,common_name,date,monthly_bird_count,log_monthly_bird_count
0,Acadian Flycatcher,2023-05-01,1.0000,0.6931
1,Acadian Flycatcher,2024-05-01,1.0000,0.6931
2,Acadian Flycatcher,2024-06-01,1.0000,0.6931
3,Acorn Woodpecker,2021-01-01,9.0000,2.3026
4,Acorn Woodpecker,2021-02-01,6.0000,1.9459
5,Acorn Woodpecker,2021-03-01,2.0000,1.0986
6,Acorn Woodpecker,2021-05-01,3.0000,1.3863
7,Acorn Woodpecker,2021-06-01,16.0000,2.8332
8,Acorn Woodpecker,2021-07-01,15.0000,2.7726
9,Acorn Woodpecker,2021-08-01,18.0000,2.9444


### Snapshot Before Prevalence Pivot

In [174]:
species_monthly.head(10)

,common_name,date,monthly_bird_count,log_monthly_bird_count
0,Acadian Flycatcher,2023-05-01,1.0000,0.6931
1,Acadian Flycatcher,2024-05-01,1.0000,0.6931
2,Acadian Flycatcher,2024-06-01,1.0000,0.6931
3,Acorn Woodpecker,2021-01-01,9.0000,2.3026
4,Acorn Woodpecker,2021-02-01,6.0000,1.9459
5,Acorn Woodpecker,2021-03-01,2.0000,1.0986
6,Acorn Woodpecker,2021-05-01,3.0000,1.3863
7,Acorn Woodpecker,2021-06-01,16.0000,2.8332
8,Acorn Woodpecker,2021-07-01,15.0000,2.7726
9,Acorn Woodpecker,2021-08-01,18.0000,2.9444


In [175]:
species_prevalence_wide = (
    species_monthly
    .pivot(index='common_name', columns='date', values='log_monthly_bird_count')
    .fillna(0)
)

# Scaled measure of species prevalence introduces negative values where there were none before
species_prevalence_scaled = pd.DataFrame(
    StandardScaler().fit_transform(species_prevalence_wide),
    index=species_prevalence_wide.index,
    columns=species_prevalence_wide.columns
)

species_prevalence_scaled.iloc[:10, :8]

date,2021-01-01,2021-02-01,2021-03-01,2021-04-01,2021-05-01,2021-06-01,2021-07-01,2021-08-01
common_name,,,,,,,,
Acadian Flycatcher,-0.7252,-0.7075,-0.7781,-0.9609,-1.1959,-0.9999,-0.9614,-0.9789
Acorn Woodpecker,0.3430,0.2323,-0.2563,-0.9609,-0.4769,0.4286,0.3719,0.4025
African Collared-Dove,-0.7252,-0.7075,-0.7781,-0.9609,-1.1959,-0.6504,-0.9614,-0.9789
Alder Flycatcher,-0.7252,-0.7075,-0.7781,-0.9609,-0.0017,-0.9999,-0.9614,-0.4635
American Avocet,-0.7252,-0.7075,0.8372,1.6491,1.2740,1.7376,1.6701,1.6764
American Barn Owl,0.3872,0.1579,-0.0137,0.3441,0.4303,0.3307,0.0386,0.2916
American Bittern,-0.7252,-0.7075,-0.7781,0.5706,0.6175,0.8069,0.4285,0.2592
American Coot,2.4899,2.5865,1.9732,2.1877,1.7991,1.4386,1.6099,1.8494
American Crow,1.5931,1.8658,2.0872,1.3299,0.8730,1.1048,1.9173,3.1799


### Snapshot After Prevalence Scaling

In [176]:
species_prevalence_scaled.describe().T.head(10)

,count,mean,std,min,25%,50%,75%,max
date,,,,,,,,
2021-01-01,567.0000,0.0000,1.0009,-0.7252,-0.7252,-0.7252,0.8369,3.8976
2021-02-01,567.0000,0.0000,1.0009,-0.7075,-0.7075,-0.7075,0.8064,3.4005
2021-03-01,567.0000,-0.0000,1.0009,-0.7781,-0.7781,-0.7781,0.8450,3.6693
2021-04-01,567.0000,0.0000,1.0009,-0.9609,-0.9609,-0.2928,0.8516,2.9647
2021-05-01,567.0000,0.0000,1.0009,-1.1959,-1.1959,0.2421,0.8119,2.3379
2021-06-01,567.0000,0.0000,1.0009,-0.9999,-0.9999,-0.3009,0.9413,2.6745
2021-07-01,567.0000,0.0000,1.0009,-0.9614,-0.9614,-0.2947,0.9387,2.3589
2021-08-01,567.0000,0.0000,1.0009,-0.9789,-0.9789,-0.2238,0.8794,3.1799
2021-09-01,567.0000,0.0000,1.0009,-1.0026,-1.0026,-0.2355,0.8222,3.6703


In [177]:
prevalence_scores, prevalence_best_k = evaluate_kmeans_grid(species_prevalence_scaled, k_values=range(2, 9))
prevalence_model, prevalence_labels, prevalence_metrics = fit_final_kmeans(species_prevalence_scaled, prevalence_best_k)

species_prevalence_clusters = species_prevalence_scaled.copy()
species_prevalence_clusters['cluster'] = prevalence_labels
species_prevalence_clusters['avg_scaled_prevalence'] = species_prevalence_scaled.mean(axis=1)
species_prevalence_clusters['prevalence_trend'] = (
    species_prevalence_wide.iloc[:, -12:].mean(axis=1) - species_prevalence_wide.iloc[:, :12].mean(axis=1)
)

prevalence_scores

,k,silhouette_score,davies_bouldin_index
0,2,0.4960,0.7972
1,3,0.4808,0.8617
2,4,0.4566,0.9542
3,5,0.4583,0.9241
4,6,0.4310,1.0360
5,7,0.4067,1.1874
6,8,0.3941,1.2088


In [178]:
prevalence_cluster_summary = (
    species_prevalence_clusters
    .groupby('cluster')
    # Compute the following for each cluster:
    # 1. Number of species
    # 2. Average standardized (scaled) prevalence of species
    # 3. Average time trend for the species in the cluster: Generally increasing, decreasing, or stable over time
    .agg(
        species_count=('avg_scaled_prevalence', 'size'),
        mean_scaled_prevalence=('avg_scaled_prevalence', 'mean'),
        mean_trend=('prevalence_trend', 'mean')
    )
    .sort_values('mean_trend', ascending=False)
)

species_prevalence_clusters[['cluster', 'avg_scaled_prevalence', 'prevalence_trend']].sort_values('prevalence_trend', ascending=False).head(15)

date,cluster,avg_scaled_prevalence,prevalence_trend
common_name,,,
Cassia Crossbill,0,-0.5162,0.9619
Gambel's Quail,1,1.6187,0.9089
Greater Roadrunner,1,0.1971,0.8516
Yellow-billed Loon,0,-0.4781,0.8496
Northern Parula,0,-0.2312,0.7143
Sagebrush Sparrow,1,0.3218,0.7015
American Barn Owl,1,0.3149,0.7004
Muscovy Duck,0,-0.4903,0.6880
Anhinga,0,-0.8338,0.6816


The prevalence clusters separate species into shared temporal patterns such as increasing prevalence, declining prevalence, and relatively stable seasonal occurrence. Species with the largest positive `prevalence_trend` have become more prevalent in the later part of the time series, while strongly negative values indicate decline.

## 3. Typical Flight Patterns by Species

This analysis estimates flight-pattern types using the county locations recorded in `birdsong_df` across time. Each species is represented by its county-level distribution over months. That lets the model group species with similar geographic spread, county turnover, and seasonal concentration patterns. To turn those clusters into interpretable movement corridors, county centroids are estimated from the raw eBird latitude and longitude observations in `ebird_co.csv`, then aggregated into month-to-month cluster paths.

In [179]:
species_monthly_county = (
    birdsong_df
    .groupby(['common_name', 'month', 'county'], as_index=False)
    .agg(monthly_count=('bird_count', 'sum'))
)

species_monthly_county['log_monthly_count'] = np.log1p(species_monthly_county['monthly_count'])
species_monthly_county.head(10)

,common_name,month,county,monthly_count,log_monthly_count
0,Acadian Flycatcher,5,Lincoln,1.0000,0.6931
1,Acadian Flycatcher,5,Pueblo,1.0000,0.6931
2,Acadian Flycatcher,6,Pueblo,1.0000,0.6931
3,Acorn Woodpecker,1,La Plata,37.0000,3.6376
4,Acorn Woodpecker,2,La Plata,30.0000,3.4340
5,Acorn Woodpecker,3,La Plata,3.0000,1.3863
6,Acorn Woodpecker,4,La Plata,8.0000,2.1972
7,Acorn Woodpecker,5,La Plata,18.0000,2.9444
8,Acorn Woodpecker,5,Larimer,1.0000,0.6931
9,Acorn Woodpecker,5,Montezuma,1.0000,0.6931


### Snapshot Before Flight-Pattern Feature Engineering

In [180]:
species_monthly_county.head(10)

,common_name,month,county,monthly_count,log_monthly_count
0,Acadian Flycatcher,5,Lincoln,1.0000,0.6931
1,Acadian Flycatcher,5,Pueblo,1.0000,0.6931
2,Acadian Flycatcher,6,Pueblo,1.0000,0.6931
3,Acorn Woodpecker,1,La Plata,37.0000,3.6376
4,Acorn Woodpecker,2,La Plata,30.0000,3.4340
5,Acorn Woodpecker,3,La Plata,3.0000,1.3863
6,Acorn Woodpecker,4,La Plata,8.0000,2.1972
7,Acorn Woodpecker,5,La Plata,18.0000,2.9444
8,Acorn Woodpecker,5,Larimer,1.0000,0.6931
9,Acorn Woodpecker,5,Montezuma,1.0000,0.6931


In [181]:
'''
Data transformations to get metrics for measuring movement of bird species across counties over time
'''

species_county_month_wide = (
    species_monthly_county
    .assign(month_county=lambda df: 'm' + df['month'].astype(str).str.zfill(2) + '_' + df['county'].str.replace(' ', '_', regex=False))
    .pivot(index='common_name', columns='month_county', values='log_monthly_count')
    .fillna(0)
)

monthly_species_summary = (
    species_monthly_county
    .groupby(['common_name', 'month'], as_index=False)
    .agg(
        monthly_total_count=('monthly_count', 'sum'),
        counties_active=('county', 'nunique')
    )
)

county_turnover = (
    species_monthly_county
    .groupby(['common_name', 'month'])['county']
    .apply(lambda x: set(x))
    .reset_index(name='county_set')
    .sort_values(['common_name', 'month'])
)

turnover_rows = []
for species_name, species_df in county_turnover.groupby('common_name'):
    sets = list(species_df['county_set'])
    if len(sets) <= 1:
        turnover_rows.append({'common_name': species_name, 'avg_county_turnover': 0.0})
        continue

    turnovers = []
    for prev_set, curr_set in zip(sets[:-1], sets[1:]):
        union_size = len(prev_set | curr_set)
        if union_size == 0:
            turnovers.append(0.0)
        else:
            turnovers.append(1 - (len(prev_set & curr_set) / union_size))
    # avg_county_turnover: A measure of month-to-month change in county footprint
    turnover_rows.append({'common_name': species_name, 'avg_county_turnover': float(np.mean(turnovers))})

county_turnover_features = pd.DataFrame(turnover_rows)

# For each species, compute: 1. total unique counties the species appears in, 2. how many months the species appears at all
# 3. average count per species-county-month row, 4. total count across the full dataset
flight_pattern_features = (
    species_monthly_county
    .groupby('common_name', as_index=False)
    .agg(
        counties_visited=('county', 'nunique'),
        active_months=('month', 'nunique'),
        avg_monthly_count=('monthly_count', 'mean'),
        total_observed_count=('monthly_count', 'sum')
    )
    # Add average number of counties that the species is active in per month
    .merge(
        monthly_species_summary.groupby('common_name', as_index=False).agg(avg_active_counties=('counties_active', 'mean')),
        on='common_name',
        how='left'
    )
    # Add avg_county_turnover
    .merge(county_turnover_features, on='common_name', how='left')
)

flight_pattern_features['log_total_observed_count'] = np.log1p(flight_pattern_features['total_observed_count'])
flight_pattern_features = flight_pattern_features.drop(columns='total_observed_count')
flight_pattern_features.head(10)

,common_name,counties_visited,active_months,avg_monthly_count,avg_active_counties,avg_county_turnover,log_total_observed_count
0,Acadian Flycatcher,2,2,1.0000,1.5000,0.5000,1.3863
1,Acorn Woodpecker,4,12,19.2105,1.5833,0.2348,5.9026
2,African Collared-Dove,6,9,1.6667,1.3333,0.9375,3.0445
3,Alder Flycatcher,16,4,1.7826,5.7500,0.7564,3.7377
4,American Avocet,44,10,43.9704,20.3000,0.4581,9.0968
5,American Barn Owl,33,12,5.7353,14.1667,0.5020,6.8835
6,American Bittern,25,8,7.7089,9.8750,0.5552,6.4135
7,American Coot,56,12,117.0222,26.2500,0.4050,10.5150
8,American Crow,50,12,512.2253,21.0833,0.4830,11.7722
9,American Dipper,41,12,9.0142,23.4167,0.3383,7.8376


### Snapshot After Flight-Pattern Feature Engineering

In [182]:
flight_numeric_cols = [
    'counties_visited', 'active_months', 'avg_monthly_count',
    'avg_active_counties', 'avg_county_turnover', 'log_total_observed_count'
]

flight_summary_scaled = pd.DataFrame(
    StandardScaler().fit_transform(flight_pattern_features.set_index('common_name')[flight_numeric_cols]),
    index=flight_pattern_features['common_name'],
    columns=flight_numeric_cols
)

flight_distribution_scaled = pd.DataFrame(
    StandardScaler().fit_transform(species_county_month_wide),
    index=species_county_month_wide.index,
    columns=species_county_month_wide.columns
)

flight_pattern_scaled = flight_summary_scaled.join(flight_distribution_scaled, how='inner')
flight_pattern_scaled.head(10)

,counties_visited,active_months,avg_monthly_count,avg_active_counties,avg_county_turnover,log_total_observed_count,m01_Adams,m01_Alamosa,m01_Arapahoe,m01_Archuleta,m01_Baca,m01_Bent,m01_Boulder,m01_Broomfield,m01_Chaffee,m01_Cheyenne,m01_Clear_Creek,m01_Costilla,m01_Crowley,m01_Custer,m01_Delta,m01_Denver,m01_Dolores,m01_Douglas,m01_Eagle,m01_El_Paso,m01_Elbert,m01_Fremont,m01_Garfield,m01_Gilpin,m01_Grand,m01_Gunnison,m01_Hinsdale,m01_Huerfano,m01_Jackson,m01_Jefferson,m01_Kiowa,m01_Kit_Carson,m01_La_Plata,m01_Lake,m01_Larimer,m01_Las_Animas,m01_Lincoln,m01_Logan,m01_Mesa,m01_Mineral,m01_Moffat,m01_Montezuma,m01_Montrose,m01_Morgan,...,m12_Crowley,m12_Custer,m12_Delta,m12_Denver,m12_Dolores,m12_Douglas,m12_Eagle,m12_El_Paso,m12_Elbert,m12_Fremont,m12_Garfield,m12_Gilpin,m12_Grand,m12_Gunnison,m12_Huerfano,m12_Jackson,m12_Jefferson,m12_Kiowa,m12_Kit_Carson,m12_La_Plata,m12_Lake,m12_Larimer,m12_Las_Animas,m12_Lincoln,m12_Logan,m12_Mesa,m12_Mineral,m12_Moffat,m12_Montezuma,m12_Montrose,m12_Morgan,m12_Otero,m12_Ouray,m12_Park,m12_Phillips,m12_Pitkin,m12_Prowers,m12_Pueblo,m12_Rio_Blanco,m12_Rio_Grande,m12_Routt,m12_Saguache,m12_San_Juan,m12_San_Miguel,m12_Sedgwick,m12_Summit,m12_Teller,m12_Washington,m12_Weld,m12_Yuma
common_name,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Acadian Flycatcher,-1.2135,-1.4986,-0.2947,-1.0281,-0.1262,-1.5475,-0.2730,-0.1417,-0.5276,-0.2276,-0.2608,-0.2985,-0.5823,-0.3462,-0.2687,-0.0772,-0.1706,-0.1036,-0.1240,-0.1211,-0.2756,-0.4249,-0.2216,-0.3479,-0.2464,-0.5774,-0.1192,-0.3995,-0.2990,-0.1164,-0.2207,-0.1260,-0.0691,-0.1730,-0.1137,-0.5721,-0.2099,-0.1283,-0.4074,-0.0970,-0.5755,-0.1629,-0.1369,-0.1993,-0.5222,-0.1029,-0.1661,-0.4465,-0.3602,-0.1311,...,-0.1491,-0.1732,-0.2318,-0.4210,-0.1419,-0.3768,-0.2051,-0.5508,-0.1002,-0.3988,-0.2309,-0.1241,-0.1988,-0.1835,-0.2593,-0.0923,-0.6030,-0.1369,-0.1794,-0.3964,-0.0968,-0.5806,-0.2061,-0.1489,-0.2701,-0.4847,-0.0967,-0.1329,-0.4497,-0.3122,-0.2467,-0.3264,-0.2622,-0.2058,-0.1221,-0.3109,-0.2314,-0.5593,-0.1247,-0.1175,-0.1765,-0.0952,-0.0528,-0.1571,-0.1168,-0.2384,-0.1943,-0.1476,-0.4563,-0.2288
Acorn Woodpecker,-1.1150,1.0386,-0.0438,-1.0192,-1.1223,0.0526,-0.2730,-0.1417,-0.5276,-0.2276,-0.2608,-0.2985,-0.5823,-0.3462,-0.2687,-0.0772,-0.1706,-0.1036,-0.1240,-0.1211,-0.2756,-0.4249,-0.2216,-0.3479,-0.2464,-0.5774,-0.1192,-0.3995,-0.2990,-0.1164,-0.2207,-0.1260,-0.0691,-0.1730,-0.1137,-0.5721,-0.2099,-0.1283,2.7306,-0.0970,-0.5755,-0.1629,-0.1369,-0.1993,-0.5222,-0.1029,-0.1661,-0.4465,-0.3602,-0.1311,...,-0.1491,-0.1732,-0.2318,-0.4210,-0.1419,-0.3768,-0.2051,-0.5508,-0.1002,-0.3988,-0.2309,-0.1241,-0.1988,-0.1835,-0.2593,-0.0923,-0.6030,-0.1369,-0.1794,1.6732,-0.0968,-0.5806,-0.2061,-0.1489,-0.2701,-0.4847,-0.0967,-0.1329,-0.4497,-0.3122,-0.2467,-0.3264,-0.2622,-0.2058,-0.1221,-0.3109,-0.2314,-0.5593,-0.1247,-0.1175,-0.1765,-0.0952,-0.0528,-0.1571,-0.1168,-0.2384,-0.1943,-0.1476,-0.4563,-0.2288
African Collared-Dove,-1.0164,0.2774,-0.2855,-1.0459,1.5173,-0.9600,-0.2730,-0.1417,-0.5276,-0.2276,-0.2608,-0.2985,0.0388,-0.3462,-0.2687,-0.0772,-0.1706,-0.1036,-0.1240,-0.1211,-0.2756,-0.4249,-0.2216,-0.3479,-0.2464,-0.5774,-0.1192,-0.3995,-0.2990,-0.1164,-0.2207,-0.1260,-0.0691,-0.1730,-0.1137,-0.5721,-0.2099,-0.1283,-0.4074,-0.0970,-0.5755,-0.1629,-0.1369,-0.1993,-0.5222,-0.1029,-0.1661,-0.4465,-0.3602,-0.1311,...,-0.1491,-0.1732,-0.2318,-0.4210,-0.1419,-0.3768,-0.2051,-0.5508,-0.1002,-0.3988,-0.2309,-0.1241,-0.1988,-0.1835,-0.2593,-0.0923,-0.6030,-0.1369,-0.1794,-0.3964,-0.0968,-0.5806,-0.2061,-0.1489,-0.2701,-0.4847,-0.0967,-0.1329,-0.4497,-0.3122,-0.2467,-0.3264,-0.2622,-0.2058,-0.1221,-0.3109,-0.2314,-0.5593,-0.1247,-0.1175,-0.1765,-0.0952,-0.0528,-0.1571,-0.1168,-0.2384,-0.1943,-0.1476,-0.4563,-0.2288
Alder Flycatcher,-0.5235,-0.9912,-0.2839,-0.5743,0.8370,-0.7144,-0.2730,-0.1417,-0.5276,-0.2276,-0.2608,-0.2985,-0.5823,-0.3462,-0.2687,-0.0772,-0.1706,-0.1036,-0.1240,-0.1211,-0.2756,-0.42

In [183]:
flight_scores, flight_best_k = evaluate_kmeans_grid(flight_pattern_scaled[flight_numeric_cols], k_values=range(2, 9))
flight_model, flight_labels, flight_metrics = fit_final_kmeans(flight_pattern_scaled[flight_numeric_cols], flight_best_k)

flight_pattern_scaled['cluster'] = flight_labels
flight_scores

,k,silhouette_score,davies_bouldin_index
0,2,0.4508,0.9134
1,3,0.4816,0.7172
2,4,0.5054,0.6172
3,5,0.4406,0.7538
4,6,0.3818,0.9250
5,7,0.3642,0.9253
6,8,0.3645,0.9337


In [184]:
# Identify clusters

flight_cluster_summary = (
    flight_pattern_features
    .assign(cluster=flight_labels)
    .groupby('cluster')
    .agg(
        species_count=('common_name', 'size'),
        counties_visited=('counties_visited', 'mean'),
        active_months=('active_months', 'mean'),
        avg_active_counties=('avg_active_counties', 'mean'),
        avg_county_turnover=('avg_county_turnover', 'mean'),
        avg_monthly_count=('avg_monthly_count', 'mean')
    )
    .sort_values(['avg_county_turnover', 'counties_visited'], ascending=False)
)

cluster_example_species = (
    flight_pattern_features
    .assign(cluster=flight_labels)
    .sort_values(['cluster', 'active_months', 'counties_visited', 'avg_monthly_count'], ascending=[True, False, False, False])
    .groupby('cluster')['common_name']
    .apply(lambda s: list(s.head(5)))
    .to_dict()
)

### Movement Behavior for Each Cluster

The analysis below combines the clustering output with county-level corridor inference so each cluster can be interpreted as a movement type. We map counties to their respective centroids, then find the coordinates for monthly cluster centers which are computed as weighted averages of the centroids of the cluster's counties, where larger weights are assigned to counties wih more bird observations. Then we compare the monthly cluster centers across time to identify the clusters' movement paths and overall directions. The final results are rendered at the end in markdown.

In [185]:
raw_ebird_df = pd.read_csv('../data/ebird_co.csv')

county_centroids = (
    raw_ebird_df
    .dropna(subset=['subnational2Name', 'lat', 'lng'])
    .groupby('subnational2Name', as_index=False)
    .agg(
        county_lat=('lat', 'mean'),
        county_lng=('lng', 'mean')
    )
    .rename(columns={'subnational2Name': 'county'})
)

flight_cluster_assignments = flight_pattern_features[['common_name']].copy()
flight_cluster_assignments['cluster'] = flight_labels

cluster_monthly_county_paths = (
    species_monthly_county
    .merge(flight_cluster_assignments, on='common_name', how='inner')
    .merge(county_centroids, on='county', how='left')
    .dropna(subset=['county_lat', 'county_lng'])
    .groupby(['cluster', 'month', 'county', 'county_lat', 'county_lng'], as_index=False)
    .agg(cluster_monthly_count=('monthly_count', 'sum'))
)

cluster_monthly_centers = (
    cluster_monthly_county_paths
    .groupby(['cluster', 'month'], as_index=False)
    .apply(
        lambda df: pd.Series({
            'weighted_lat': np.average(df['county_lat'], weights=df['cluster_monthly_count']),
            'weighted_lng': np.average(df['county_lng'], weights=df['cluster_monthly_count']),
            'total_cluster_count': df['cluster_monthly_count'].sum(),
            'top_counties': ' -> '.join(
                df.sort_values('cluster_monthly_count', ascending=False)['county'].head(3)
            )
        })
        , include_groups=False
    )
    .reset_index(drop=True)
    .sort_values(['cluster', 'month'])
)

def describe_direction(lat_change, lng_change, threshold=0.15):
    north_south = ''
    east_west = ''

    if lat_change > threshold:
        north_south = 'north'
    elif lat_change < -threshold:
        north_south = 'south'

    if lng_change > threshold:
        east_west = 'east'
    elif lng_change < -threshold:
        east_west = 'west'

    if north_south and east_west:
        return f'{north_south}{east_west}'
    if north_south:
        return north_south
    if east_west:
        return east_west
    return 'stable'

movement_rows = []
for cluster_id, cluster_df in cluster_monthly_centers.groupby('cluster'):
    cluster_df = cluster_df.sort_values('month').reset_index(drop=True)
    for i in range(len(cluster_df) - 1):
        start_row = cluster_df.iloc[i]
        end_row = cluster_df.iloc[i + 1]
        movement_rows.append({
            'cluster': cluster_id,
            'start_month': int(start_row['month']),
            'end_month': int(end_row['month']),
            'start_counties': start_row['top_counties'],
            'end_counties': end_row['top_counties'],
            'lat_change': end_row['weighted_lat'] - start_row['weighted_lat'],
            'lng_change': end_row['weighted_lng'] - start_row['weighted_lng'],
            'direction': describe_direction(
                end_row['weighted_lat'] - start_row['weighted_lat'],
                end_row['weighted_lng'] - start_row['weighted_lng']
            )
        })

cluster_direction_steps = pd.DataFrame(movement_rows)

cluster_path_summary = (
    cluster_direction_steps
    .groupby('cluster', as_index=False)
    .agg(
        first_active_month=('start_month', 'min'),
        last_active_month=('end_month', 'max'),
        representative_path=('start_counties', lambda s: ' | '.join(pd.Series(s).drop_duplicates().head(4))),
        dominant_directions=('direction', lambda s: ' -> '.join(pd.Series(s).replace('stable', np.nan).dropna().head(6)))
    )
)

overall_direction = (
    cluster_direction_steps
    .groupby('cluster', as_index=False)
    .agg(total_lat_change=('lat_change', 'sum'), total_lng_change=('lng_change', 'sum'))
)
overall_direction['overall_direction'] = overall_direction.apply(
    lambda row: describe_direction(row['total_lat_change'], row['total_lng_change'], threshold=0.3),
    axis=1
)

cluster_path_summary = cluster_path_summary.merge(overall_direction[['cluster', 'overall_direction']], on='cluster', how='left')
cluster_path_summary['dominant_directions'] = cluster_path_summary['dominant_directions'].replace('', 'stable')

month_names = {
    1: 'January', 2: 'February', 3: 'March', 4: 'April', 5: 'May', 6: 'June',
    7: 'July', 8: 'August', 9: 'September', 10: 'October', 11: 'November', 12: 'December'
}

# Function to convert numeric cluster metrics into plain-language traits
def classify_movement_traits(row):
    traits = []
    if row['avg_county_turnover'] >= 0.65:
        traits.append('high county turnover')
    elif row['avg_county_turnover'] >= 0.35:
        traits.append('moderate county turnover')
    else:
        traits.append('low county turnover')

    if row['active_months'] >= 9:
        traits.append('year-round or near year-round activity')
    elif row['active_months'] >= 5:
        traits.append('clear seasonal presence')
    else:
        traits.append('short or sporadic seasonal windows')

    if row['counties_visited'] >= 30:
        traits.append('broad statewide footprint')
    elif row['counties_visited'] >= 8:
        traits.append('regional multi-county footprint')
    else:
        traits.append('narrow county footprint')

    return ', '.join(traits)

# Use the path summary as a lookup table when writing the narrative for each cluster.
cluster_path_lookup = cluster_path_summary.set_index('cluster')
cluster_lines = ['### Cluster Interpretations']

# Render the final writeup in ascending cluster order for readability.
for _, summary_row in flight_cluster_summary.reset_index().sort_values('cluster').iterrows():
    cluster_num = int(summary_row['cluster'])
    path_row = cluster_path_lookup.loc[cluster_num]
    # Pull representative species examples and convert the metrics into plain-language traits.
    examples = ', '.join(f'**{species}**' for species in cluster_example_species.get(cluster_num, []))
    movement_traits = classify_movement_traits(summary_row)
    month_span = f"{month_names[int(path_row['first_active_month'])]} to {month_names[int(path_row['last_active_month'])]}"

    # Build one short paragraph block per cluster for the markdown report.
    cluster_lines.append(f"#### Cluster {cluster_num}")
    cluster_lines.append(
        f"This cluster contains **{int(summary_row['species_count'])} species** and is characterized by {movement_traits}. "
        f"On average, species in this cluster are active across **{summary_row['active_months']:.1f} months**, visit **{summary_row['counties_visited']:.1f} counties**, and show **{summary_row['avg_county_turnover']:.2f}** month-to-month county turnover."
    )
    cluster_lines.append(
        f"Across {month_span}, the representative county path is **{path_row['representative_path']}**. "
        f"The dominant month-to-month movement directions are **{path_row['dominant_directions']}**, and the overall drift is **{path_row['overall_direction']}**."
    )
    cluster_lines.append(f"Example species in this cluster include {examples}.")
    cluster_lines.append('')

display(Markdown('\n'.join(cluster_lines)))


### Cluster Interpretations
#### Cluster 0
This cluster contains **200 species** and is characterized by high county turnover, clear seasonal presence, regional multi-county footprint. On average, species in this cluster are active across **6.1 months**, visit **11.5 counties**, and show **0.75** month-to-month county turnover.
Across January to December, the representative county path is **Pueblo -> Jefferson -> Larimer | Pueblo -> Denver -> Larimer | Jefferson -> Larimer -> Denver | Yuma -> Washington -> Montezuma**. The dominant month-to-month movement directions are **east -> southeast -> southwest -> west -> east -> northeast**, and the overall drift is **stable**.
Example species in this cluster include **Boreal Owl**, **Red-necked Grebe**, **Pacific Loon**, **White-winged Crossbill**, **Helmeted Guineafowl**.

#### Cluster 1
This cluster contains **285 species** and is characterized by moderate county turnover, year-round or near year-round activity, broad statewide footprint. On average, species in this cluster are active across **10.7 months**, visit **43.6 counties**, and show **0.52** month-to-month county turnover.
Across January to December, the representative county path is **Larimer -> Jefferson -> Boulder | Larimer -> Boulder -> Jefferson | Larimer -> Boulder -> Lincoln | Boulder -> Larimer -> Arapahoe**. The dominant month-to-month movement directions are **northeast -> west -> east -> west**, and the overall drift is **stable**.
Example species in this cluster include **Northern Harrier**, **Prairie Falcon**, **Sharp-shinned Hawk**, **Golden Eagle**, **American Pipit**.

#### Cluster 2
This cluster contains **76 species** and is characterized by low county turnover, short or sporadic seasonal windows, narrow county footprint. On average, species in this cluster are active across **1.7 months**, visit **1.2 counties**, and show **0.03** month-to-month county turnover.
Across January to December, the representative county path is **Montezuma -> Arapahoe -> Larimer | Larimer -> Montezuma -> Arapahoe | Montezuma -> Larimer -> Baca | El Paso -> Larimer -> Montezuma**. The dominant month-to-month movement directions are **northeast -> southeast -> northeast -> east -> northwest -> southwest**, and the overall drift is **east**.
Example species in this cluster include **American Wigeon x Mallard (hybrid)**, **Steller's Jay x Woodhouse's Scrub-Jay (hybrid)**, **Canyon x Spotted Towhee (hybrid)**, **Crissal Thrasher**, **Bufflehead x Common Goldeneye (hybrid)**.

#### Cluster 3
This cluster contains **6 species** and is characterized by moderate county turnover, year-round or near year-round activity, broad statewide footprint. On average, species in this cluster are active across **11.5 months**, visit **46.0 counties**, and show **0.47** month-to-month county turnover.
Across January to December, the representative county path is **Kit Carson -> Baca -> Weld | Bent -> Boulder -> Mesa | Rio Grande -> Conejos -> Denver | Arapahoe -> Larimer -> Weld**. The dominant month-to-month movement directions are **southwest -> southwest -> northeast -> west -> east -> north**, and the overall drift is **northwest**.
Example species in this cluster include **Snow Goose**, **Sandhill Crane**, **American Crow**, **Ring-billed Gull**, **Cackling Goose**.


## Model Evaluation

In [186]:
evaluation_summary = pd.DataFrame([
    {
        'analysis': 'Seasonal hotspots',
        'best_k': ', '.join(f"{row.season}:{row.best_k}" for row in season_hotspot_metrics_df.itertuples()),
        'silhouette_score': season_hotspot_metrics_df['silhouette_score'].mean(),
        'davies_bouldin_index': season_hotspot_metrics_df['davies_bouldin_index'].mean()
    },
    {
        'analysis': 'Species prevalence over time',
        'best_k': prevalence_best_k,
        'silhouette_score': prevalence_metrics['silhouette_score'],
        'davies_bouldin_index': prevalence_metrics['davies_bouldin_index']
    },
    {
        'analysis': 'Flight-pattern types',
        'best_k': flight_best_k,
        'silhouette_score': flight_metrics['silhouette_score'],
        'davies_bouldin_index': flight_metrics['davies_bouldin_index']
    }
])

evaluation_summary

,analysis,best_k,silhouette_score,davies_bouldin_index
0,Seasonal hotspots,"Autumn:2, Spring:3, Summer:3, Winter:2",0.5944,0.6862
1,Species prevalence over time,2,0.4960,0.7972
2,Flight-pattern types,4,0.5054,0.6172


## Challenges and Solutions

- **Challenge:** Bird counts span very different scales across counties and species.  
  **Solution:** Standardization and `log1p` transforms were used to keep large counts from dominating Euclidean distance.

- **Challenge:** The hotspot question is geographic, but `birdsong_df` is observation-level rather than county-season labeled.  
  **Solution:** The data was aggregated to county-season summaries that preserve abundance, diversity, and environmental context.

- **Challenge:** Flight paths are not directly observed, and the cleaned modeling table is aggregated at the county-month level rather than as individual tracked trajectories.  
  **Solution:** County centroids were estimated from the raw eBird latitude and longitude observations in `ebird_co.csv`, then monthly cluster centers were computed from county-level bird counts to infer likely corridors and overall movement direction.

- **Challenge:** Choosing `k` is subjective in unsupervised learning.  
  **Solution:** Multiple values of `k` were compared using both Silhouette Score and Davies-Bouldin Index.

## Final Interpretation Notes

- Higher Silhouette Score is better because it indicates tighter, better-separated clusters.
- Lower Davies-Bouldin Index is better because it indicates lower within-cluster dispersion relative to between-cluster separation.
- If either metric is weak for a task, the clusters may still be useful for exploration, but they should be interpreted as soft groupings rather than definitive ecological classes.